# 03 - Validation diagnostics

Three questions that a single RMSE number cannot answer:

1. **Is plain linear regression (OLS) a good reference point?** (the taught method, with no penalty)
2. **Is each model overfitting?** Compare error on the training engines with error on unseen validation engines.
3. **Where are the errors?** Plot error against true RUL. Errors near failure matter most.

Everything here uses the same engine-grouped folds as notebook 02. The test set is not used.
Cells marked **Your observation** are for you to complete.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().resolve().parent))

from src.data_loader import add_rul, load_fd001
from src.evaluation import cross_validate, out_of_fold_predictions
from src.features import get_feature_columns
from src.models import baseline_models

train, _, _ = load_fd001()
train = add_rul(train)
CAP = 125
FEATURES = get_feature_columns()
models = baseline_models()

INCLUDE_RANDOM_FOREST = False   # set True to include it (slow: minutes)
names = ["mean", "linear", "ridge", "hist_gradient_boosting"]
if INCLUDE_RANDOM_FOREST:
    names.append("random_forest")
print(names)

## 2. Training error vs validation error

`train_rmse` is the error on the engines the model was fitted to. `rmse` is the error on engines it never saw.
A small `train_rmse` with a much larger `rmse` means the model has memorised noise (overfitting,
your notes section 5.1). A large error on both means it is too simple to capture the pattern (underfitting).

In [ ]:
rows = {}
for name in names:
    res = cross_validate(models[name], train, FEATURES, cap=CAP)
    rows[name] = {
        "train_rmse": res["train_rmse"].mean(),
        "val_rmse": res["rmse"].mean(),
        "val_rmse_std": res["rmse"].std(),
        "gap (val - train)": res["rmse"].mean() - res["train_rmse"].mean(),
        "ratio (val / train)": res["rmse"].mean() / res["train_rmse"].mean(),
    }
gap_table = pd.DataFrame(rows).T.round(2)
gap_table

**Your observation:** Which models fit the training engines much better than unseen engines?
Which look underfit (high error on both)? How does OLS compare with ridge? If they are almost
identical, what does that say about how much regularisation matters with 14 features and ~16,000 rows?
Link your answer to the overfit/underfit language in your notes.

## 3. Where are the errors? (out-of-fold predictions)

Every training row is predicted by a model that never saw that row's engine. Error = predicted - true,
so **positive means predicted too much remaining life (a late, dangerous prediction)** and negative
means too early.

In [ ]:
oof = {name: out_of_fold_predictions(models[name], train, FEATURES, cap=CAP)
       for name in ["linear", "hist_gradient_boosting"]}
oof["hist_gradient_boosting"].head()

### 3a. Predicted vs true RUL

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
for ax, (name, df) in zip(axes, oof.items()):
    ax.scatter(df["true"], df["pred"], s=3, alpha=0.15)
    ax.plot([0, CAP], [0, CAP], "k--", lw=1)
    ax.set_title(name)
    ax.set_xlabel(f"True RUL (capped at {CAP})")
axes[0].set_ylabel("Out-of-fold predicted RUL")
fig.tight_layout()
plt.show()

### 3b. Error against true RUL

In [ ]:
bins = [-1, 25, 50, 75, 100, CAP]
labels = ["0-25", "26-50", "51-75", "76-100", f"101-{CAP}"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (name, df) in zip(axes, oof.items()):
    ax.scatter(df["true"], df["error"], s=3, alpha=0.12)
    ax.axhline(0, color="k", lw=1)
    ax.set_title(name)
    ax.set_xlabel("True RUL (cycles)")
axes[0].set_ylabel("Error = predicted - true (cycles)")
fig.suptitle("Positive = predicted too much life (late)", y=1.02)
fig.tight_layout()
plt.show()

### 3c. Error by RUL band

In [ ]:
def band_summary(df):
    band = pd.cut(df["true"], bins=bins, labels=labels)
    g = df.groupby(band, observed=True)["error"]
    return pd.DataFrame({
        "n": g.size(),
        "RMSE": g.apply(lambda e: float(np.sqrt(np.mean(e ** 2)))),
        "MAE": g.apply(lambda e: float(np.mean(np.abs(e)))),
        "mean error (bias)": g.mean(),
    })

for name, df in oof.items():
    print(name)
    display(band_summary(df).round(2))

**Your observation:** In which RUL band is each model most accurate and least accurate? Is the bias
(mean error) positive or negative in the bands close to failure? A model that is accurate on average but
late near failure is a problem for maintenance decisions. Are the large errors that raise RMSE above MAE
concentrated in particular bands? Note which model is better *in the 0-50 band*, even if another is
better overall.

### 3d. Per-engine view: are some engines much harder than others?

In [ ]:
per_engine = (oof["hist_gradient_boosting"]
              .groupby("engine")["error"]
              .agg(rmse=lambda e: float(np.sqrt(np.mean(e ** 2))), bias="mean")
              .sort_values("rmse", ascending=False))
display(per_engine.head(8).round(1))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(per_engine["rmse"], bins=20, edgecolor="black")
ax.set_xlabel("Per-engine RMSE (cycles)")
ax.set_ylabel("Number of engines")
ax.set_title("hist_gradient_boosting: error varies by engine")
plt.show()

**Your observation:** Is the error spread fairly evenly across engines, or do a few engines dominate?
Pick the worst engine and plot its sensors (as in notebook 01) to see if something about it is unusual.
What does this suggest about how well 100 training engines represent the fleet?

## Summary (fill in)

| Question | Finding | Evidence |
| --- | --- | --- |
| Is OLS adequate vs ridge? | | |
| Which models overfit? | | |
| Where is error highest (RUL band)? | | |
| Early or late bias near failure? | | |